# Create violin plot cellwise RA-AP ratio

This notebook creates a violin plot that displays the ratio of RA-AP spikes per cell.

## Initialization - don't change

In [ ]:
import sys
sys.path.append('..')

from param_classes.workflows.input_output import InputDataParams
from workflows.run_util.io import collect_paths_from_table

data_params = InputDataParams()

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [ ]:
palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'AIS-AP': "#3150B7",
        'RA-AP': '#00ff00',
        'UNKNOWN': '#000000b2'
    }
}

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [ ]:
MAT_DIR = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo'
PROCESSED_DIR = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed'

In [ ]:
data_params.raw_data.path= MAT_DIR
data_params.raw_data.extension='mat'
data_params.spike_features.path = f'{PROCESSED_DIR}\\SpikeFeatures'
data_params.spike_features.extension = 'csv'

In [ ]:
input_param = {
    'mode': 'table',
    'table_path': f'{PROCESSED_DIR}\\data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
output_path = f'{PROCESSED_DIR}\\RaAp_Ratio_Plot'

In [ ]:
spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']

## Plot parameters

List of the groups in the order of display.

In [ ]:
group_list = [
    'FSI_L2-3',
    'RSI_L2-3',
    'PYR_L2-3',
    'FSI_L4-6',
    'RSI_L4-6',
    'PYR_L4-6'
]

`alpha`: transparency of the violins

In [ ]:
alpha = 0.5

What to show inside the violins:
 - `'box'`: boxplot
 - `'point'`: points for each cell
 - `None`: nothing (violins are empty)

In [ ]:
inner_mode = 'point'

Y axis limit (should be between 0 and 1)

In [ ]:
y_limit = (0, 1.0)

## Run analysis

Collect input files according to the table.

In [ ]:
input_files = collect_paths_from_table(data_params, input_param)
print(f"{len(input_files)} files selected for analysis.")

In [ ]:
import pandas as pd


spike_count_per_type = []
for cell_info in input_files:
    cell_data = cell_info.data.load()
    cell_stats: dict = {
        'name': cell_info.name,
        'cell_type': cell_info.cell_type
    }
    for spike_type in spike_types:
        count = (cell_data.spikes['spike_type'] == spike_type).sum()
        cell_stats[spike_type] = count / len(cell_data.spikes)
    spike_count_per_type.append(cell_stats)

spike_type_stats = pd.DataFrame(spike_count_per_type)
spike_type_stats['cell_type'] = pd.Categorical(
    spike_type_stats['cell_type'],
    categories=group_list,
    ordered=True
)

In [ ]:
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 6))
sns.violinplot(
    data=spike_type_stats,
    x='cell_type',
    y='RA-AP',
    hue='cell_type',
    palette=palette['cell_type'],
    linewidth=1,
    saturation=1,
    cut=0,
    inner=inner_mode,
    alpha=alpha,
)
plt.ylim(y_limit)
plt.ylabel('RA-AP Ratio')
plt.xlabel('')
plt.title('Ratio of RA-AP per cell')

save_path = Path(output_path)
save_path.mkdir(parents=True, exist_ok=True)
with plt.rc_context({'svg.fonttype': 'none'}):
    fig.savefig(
        f'{save_path}/violin_plot.svg',
        format='svg'
    )
plt.close()